# Cleaning Data

In [1]:
import pandas as pd 
import numpy as np

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth',80)

In [3]:
# load data
df = pd.read_csv("../data/raw/beforward/raw_beforward_fast_scraped.csv")
df.head()

,vehicle_id,ref_no,title,price_usd,total_price_usd,mileage,year,engine_cc,transmission,location,model_code,steering,fuel,seats,engine_code,color,drive,doors,accessories,destination_port,vehicle_url
0,15018558,Ref No. CD066265,2019 JEEP WRANGLER UNLIMITED SAHARA,16980.0,19896.0,"94,263 km",2019/9,"3,600cc",AT,Yokohama,ABA-JL36L,Right,Petrol,5,G,Gray,4WD,5,NaN,Mombasa,/jeep/wrangler/cd066265/id/15018558/?mfg_year_from=2018
1,14864798,Ref No. CC842281,2019 HONDA ACCORD,27740.0,29731.0,"55,361 km",2019/12,"1,500cc",AT,Thailand,-,Right,Petrol,5,-,Black,2WD,4,"Power Steering, A/C, Airbag, Leather Seat, Keyless Entry",Mombasa,/honda/accord/cc842281/id/14864798/?mfg_year_from=2018
2,14864891,Ref No. CC842374,2024 HONDA CITY,17190.0,18416.0,"26,954 km",2024/5,"1,000cc",AT,Thailand,-,Right,Petrol,5,-,White,2WD,4,"Power Steering, A/C, Airbag, Leather Seat, Keyless Entry",Mombasa,/honda/city/cc842374/id/14864891/?mfg_year_from=2018
3,14864903,Ref No. CC842386,2024 HONDA BR-V,22080.0,24107.0,"58,522 km",2024/7,"1,500cc",AT,Thailand,-,Right,Petrol,7,-,Black,2WD,5,"Power Steering, A/C, Airbag, Leather Seat, Keyless Entry",Mombasa,/honda/br-v/cc842386/id/14864903/?mfg_year_from=2018
4,14864910,Ref No. CC842393,2023 HONDA CR-V,40390.0,42560.0,"30,859 km",2023/11,"1,500cc",AT,Thailand,-,Right,Petrol,7,-,White,4WD,5,"Power Steering, A/C, Airbag, Leather Seat, Keyless Entry",Mombasa,/honda/cr-v/cc842393/id/14864910/?mfg_year_from=2018


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24775 entries, 0 to 24774
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   vehicle_id        24775 non-null  int64  
 1   ref_no            24775 non-null  str    
 2   title             24775 non-null  str    
 3   price_usd         24744 non-null  float64
 4   total_price_usd   24568 non-null  float64
 5   mileage           24775 non-null  str    
 6   year              24775 non-null  str    
 7   engine_cc         24552 non-null  str    
 8   transmission      24775 non-null  str    
 9   location          24775 non-null  str    
 10  model_code        24775 non-null  str    
 11  steering          24775 non-null  str    
 12  fuel              24775 non-null  str    
 13  seats             24775 non-null  str    
 14  engine_code       24775 non-null  str    
 15  color             24775 non-null  str    
 16  drive             24775 non-null  str    
 17  door

In [5]:
# exact duplicates by vehicle_id
# df[df.duplicated("vehicle_id", keep=False)]

# # duplicates by ref number
# df[df.duplicated("ref_no", keep=False)]

# # duplicates by vehicle URL
# df[df.duplicated("vehicle_url", keep=False)]

### Clean name column

In [6]:
parts = df["title"].str.split(" ", n=2, expand=True)

df['make'] = parts[1]
df['model'] = parts[2]

KNOWN_MAKES = [
    "MERCEDES-BENZ",
    "LAND ROVER",
    "ALFA ROMEO",
    "ASTON MARTIN",
    "ROLLS-ROYCE",
    "VOLKSWAGEN",
    "MITSUBISHI",
    "CHEVROLET",
    "LAMBORGHINI",
    "MASERATI",
    "DAIHATSU",
    "TOYOTA",
    "SUBARU",
    "SUZUKI",
    "NISSAN",
    "PORSCHE",
    "PEUGEOT",
    "RENAULT",
    "TESLA",
    "LEXUS",
    "HONDA",
    "VOLVO",
    "MAZDA",
    "JEEP",
    "BMW",
    "AUDI",
    "MINI",
    "FIAT",
    "ISUZU",
]

In [7]:
def extract_make(name):
    name = name.upper()

    for make in KNOWN_MAKES:
        if make in name:
            return make

    return "OTHER"

df["make"] = df["title"].apply(extract_make)

In [8]:
df[['make','model']]

,make,model
0,JEEP,WRANGLER UNLIMITED SAHARA
1,HONDA,ACCORD
2,HONDA,CITY
3,HONDA,BR-V
4,HONDA,CR-V
...,...,...
24770,ISUZU,D-MAX 3.0 SX DOUBLE CAB
24771,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB
24772,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB
24773,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB


In [12]:
df['make'].unique()

<StringArray>
[         'JEEP',         'HONDA',         'OTHER',         'ISUZU',
    'MITSUBISHI',        'NISSAN',        'TOYOTA',         'MAZDA',
           'BMW',          'MINI', 'MERCEDES-BENZ',    'VOLKSWAGEN',
          'AUDI',        'SUZUKI',        'SUBARU',      'DAIHATSU',
          'FIAT',         'LEXUS',       'RENAULT',         'VOLVO',
    'LAND ROVER',         'TESLA',       'PEUGEOT',       'PORSCHE',
      'MASERATI',    'ALFA ROMEO',  'ASTON MARTIN',     'CHEVROLET',
   'LAMBORGHINI']
Length: 29, dtype: str

### Clean mileage column

In [15]:
df["mileage"]

0        94,263 km
1        55,361 km
2        26,954 km
3        58,522 km
4        30,859 km
           ...    
24770            -
24771            -
24772            -
24773            -
24774            -
Name: mileage, Length: 24775, dtype: str

In [16]:
df["mileage_km"] = df["mileage"].str.replace(",", "", regex=False).str.replace("km", "", regex=False)

In [17]:
df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")

In [18]:
df["mileage_km"]

0        94263.0
1        55361.0
2        26954.0
3        58522.0
4        30859.0
          ...   
24770        NaN
24771        NaN
24772        NaN
24773        NaN
24774        NaN
Name: mileage_km, Length: 24775, dtype: float64

In [19]:
df["mileage_km"].isnull().sum()

np.int64(1138)

### Clean engine_cc columns

In [20]:
df['engine_cc']

0        3,600cc
1        1,500cc
2        1,000cc
3        1,500cc
4        1,500cc
          ...   
24770    3,000cc
24771    3,000cc
24772    3,000cc
24773    3,000cc
24774    3,000cc
Name: engine_cc, Length: 24775, dtype: str

In [21]:
df["engine_cc"] = (
    df["engine_cc"]\
    .str.replace(",", "", regex=False)\
    .str.replace("cc", "", regex=False)
)

In [22]:
df["engine_cc"] = pd.to_numeric(df["engine_cc"], errors="coerce")

In [23]:
df['engine_cc']

0        3600.0
1        1500.0
2        1000.0
3        1500.0
4        1500.0
          ...  
24770    3000.0
24771    3000.0
24772    3000.0
24773    3000.0
24774    3000.0
Name: engine_cc, Length: 24775, dtype: float64

### Clean year column

In [24]:
df['year']

0         2019/9
1        2019/12
2         2024/5
3         2024/7
4        2023/11
          ...   
24770     2026/6
24771     2026/1
24772     2026/4
24773     2025/5
24774     2025/5
Name: year, Length: 24775, dtype: str

In [25]:
df["year"] = df["year"].str.extract(r"(\d{4})").astype(int)
df["year"]

0        2019
1        2019
2        2024
3        2024
4        2023
         ... 
24770    2026
24771    2026
24772    2026
24773    2025
24774    2025
Name: year, Length: 24775, dtype: int64

### Clean price columns

In [26]:
df['price_usd']

0        16980.0
1        27740.0
2        17190.0
3        22080.0
4        40390.0
          ...   
24770    41260.0
24771    50090.0
24772    54520.0
24773    51910.0
24774    58920.0
Name: price_usd, Length: 24775, dtype: float64

In [27]:
df["price_usd"] = pd.to_numeric(
    df["price_usd"]
    .astype(str)
    .str.replace("US$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

In [28]:
df['price_usd']

0        16980.0
1        27740.0
2        17190.0
3        22080.0
4        40390.0
          ...   
24770    41260.0
24771    50090.0
24772    54520.0
24773    51910.0
24774    58920.0
Name: price_usd, Length: 24775, dtype: float64

### Clean accessories column

In [29]:
df["accessories"]

0                                                             NaN
1        Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
2        Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
3        Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
4        Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
                                   ...                           
24770    Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
24771     Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels
24772     Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels
24773      Power Steering, A/C, Airbag, Leather Seat, Back Camera
24774    Power Steering, A/C, Airbag, Leather Seat, Keyless Entry
Name: accessories, Length: 24775, dtype: str

In [30]:
df["accessories"] =df["accessories"].fillna("").str.split(", ")

In [31]:
df["accessories"]

0                                                                []
1        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
2        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
3        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
4        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
                                    ...                            
24770    [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
24771     [Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]
24772     [Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]
24773      [Power Steering, A/C, Airbag, Leather Seat, Back Camera]
24774    [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
Name: accessories, Length: 24775, dtype: object

### Clean ref_no column

In [32]:
df['ref_no']

0        Ref No. CD066265
1        Ref No. CC842281
2        Ref No. CC842374
3        Ref No. CC842386
4        Ref No. CC842393
               ...       
24770    Ref No. CD219642
24771    Ref No. CD219641
24772    Ref No. CD219640
24773    Ref No. CD219639
24774    Ref No. CD219638
Name: ref_no, Length: 24775, dtype: str

In [33]:
df["ref_no"] = df["ref_no"].str.replace("Ref No. ", "", regex=False)

In [34]:
df['ref_no']

0        CD066265
1        CC842281
2        CC842374
3        CC842386
4        CC842393
           ...   
24770    CD219642
24771    CD219641
24772    CD219640
24773    CD219639
24774    CD219638
Name: ref_no, Length: 24775, dtype: str

### Location

In [35]:
df['location']

0         Yokohama
1         Thailand
2         Thailand
3         Thailand
4         Thailand
           ...    
24770    Australia
24771    Australia
24772    Australia
24773    Australia
24774    Australia
Name: location, Length: 24775, dtype: str

In [36]:
df['location'].unique()

<StringArray>
[      'Yokohama',       'Thailand',         'Nagoya',         'Kyushu',
          'Osaka',       'Hokkaido',           'Kobe',         'Toyama',
        'Niigata', 'United Kingdom',          'Kyoto',      'Australia',
      'Singapore',          'Dubai',   'South Africa']
Length: 15, dtype: str

### Add shipping_cost

In [37]:
df["shipping_cost"] = df["total_price_usd"] - df["price_usd"]
df["shipping_cost"] 

0        2916.0
1        1991.0
2        1226.0
3        2027.0
4        2170.0
          ...  
24770    4785.0
24771    4685.0
24772    4685.0
24773    4685.0
24774    4685.0
Name: shipping_cost, Length: 24775, dtype: float64

### Add car_age column

In [38]:
df["car_age"] = 2026 - df["year"]
df['car_age']

0        7
1        7
2        2
3        2
4        3
        ..
24770    0
24771    0
24772    0
24773    1
24774    1
Name: car_age, Length: 24775, dtype: int64

### Clean transmission columns

In [39]:
df['transmission']

0        AT
1        AT
2        AT
3        AT
4        AT
         ..
24770    AT
24771    AT
24772    AT
24773    AT
24774    AT
Name: transmission, Length: 24775, dtype: str

In [40]:
df["transmission"] = df["transmission"].str.strip()

In [41]:
df["transmission_type"] = df["transmission"].replace({
    "AT": "Automatic",
    "CVT": "Automatic",
    "Semi AT": "Semi-Automatic",
    "MT": "Manual"
})

In [42]:
df["transmission_type"] 

0        Automatic
1        Automatic
2        Automatic
3        Automatic
4        Automatic
           ...    
24770    Automatic
24771    Automatic
24772    Automatic
24773    Automatic
24774    Automatic
Name: transmission_type, Length: 24775, dtype: str

### Clean model_code column

In [43]:
df['model_code']

0        ABA-JL36L
1                -
2                -
3                -
4                -
           ...    
24770           SX
24771         LS-U
24772         LS-U
24773         LS-U
24774         LS-U
Name: model_code, Length: 24775, dtype: str

In [44]:
df['model_code'] = df['model_code'].replace('-', None)
df['model_code']

0        ABA-JL36L
1              NaN
2              NaN
3              NaN
4              NaN
           ...    
24770           SX
24771         LS-U
24772         LS-U
24773         LS-U
24774         LS-U
Name: model_code, Length: 24775, dtype: str

In [45]:
df['model_code']

0        ABA-JL36L
1              NaN
2              NaN
3              NaN
4              NaN
           ...    
24770           SX
24771         LS-U
24772         LS-U
24773         LS-U
24774         LS-U
Name: model_code, Length: 24775, dtype: str

### Clean steering column

In [46]:
df['steering'].str.strip()

0        Right
1        Right
2        Right
3        Right
4        Right
         ...  
24770    Right
24771    Right
24772    Right
24773    Right
24774    Right
Name: steering, Length: 24775, dtype: str

### Clean seats column

In [47]:
df['seats']

0        5
1        5
2        5
3        7
4        7
        ..
24770    5
24771    5
24772    4
24773    5
24774    5
Name: seats, Length: 24775, dtype: str

In [48]:
df["seats"].unique()

<StringArray>
[   '5',    '7',   '11',    '4',    '2',    '6',   '10',    '8',    '3',
  'ASK',   '14', '5528',   '16',    '9',   '28',    '1',   '25',   '12',
   '99',   '29']
Length: 20, dtype: str

In [60]:
df[df["seats"].isin([ "99"])]# "99", "29", "28", "25"])]

,vehicle_id,ref_no,title,price_usd,total_price_usd,mileage,year,engine_cc,transmission,location,model_code,steering,fuel,seats,engine_code,color,drive,doors,accessories,destination_port,vehicle_url,make,model,mileage_km,shipping_cost,car_age,transmission_type


In [63]:
# df["seats"] = df["seats"].replace(["ASK", "-", ""], np.nan)
df["seats"] = (
    df["seats"]
    .replace(["ASK", "5228","99","", " ", "-", "NULL"], np.nan)
)

df["seats"] = pd.to_numeric(df["seats"], errors="coerce")

In [64]:
df['seats']

0        5.0
1        5.0
2        5.0
3        7.0
4        7.0
        ... 
24770    5.0
24771    5.0
24772    4.0
24773    5.0
24774    5.0
Name: seats, Length: 24775, dtype: float64

### Clean engine_code columns

In [65]:
df['engine_code']

0        G
1        -
2        -
3        -
4        -
        ..
24770    -
24771    -
24772    -
24773    -
24774    -
Name: engine_code, Length: 24775, dtype: str

In [66]:
df["engine_code"] = df["engine_code"].replace("-", np.nan)
df['engine_code']

0          G
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
24770    NaN
24771    NaN
24772    NaN
24773    NaN
24774    NaN
Name: engine_code, Length: 24775, dtype: str

In [67]:
df['engine_code'].isnull().sum()

np.int64(23472)

### Color column

In [70]:
df['color']

0          Gray
1         Black
2         White
3         Black
4         White
          ...  
24770     White
24771    Silver
24772    Silver
24773     Black
24774      Gray
Name: color, Length: 24775, dtype: str

In [71]:
df['color'].unique()

<StringArray>
[  'Gray',  'Black',  'White',  'Brown',  'Other',    'Red',   'Blue',
  'Pearl', 'Orange', 'Silver',  'Green',   'Gold', 'Maroon',  'Beige',
 'Yellow', 'Purple', 'Bronze',   'Pink']
Length: 18, dtype: str

In [72]:
df['drive']

0        4WD
1        2WD
2        2WD
3        2WD
4        4WD
        ... 
24770    4WD
24771    4WD
24772    4WD
24773    4WD
24774    4WD
Name: drive, Length: 24775, dtype: str

### Clean doors columns

In [73]:
df['doors']

0        5
1        4
2        4
3        5
4        5
        ..
24770    4
24771    4
24772    4
24773    4
24774    4
Name: doors, Length: 24775, dtype: str

In [75]:
df['doors'].unique()

<StringArray>
['5', '4', '2', '3', 'ASK', '6', '99']
Length: 7, dtype: str

In [76]:
df["doors"] = df["doors"].replace("ASK", np.nan)
df['doors']

0        5
1        4
2        4
3        5
4        5
        ..
24770    4
24771    4
24772    4
24773    4
24774    4
Name: doors, Length: 24775, dtype: str

### Clean accessories columns

In [77]:
df['accessories']

0                                                                []
1        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
2        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
3        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
4        [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
                                    ...                            
24770    [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
24771     [Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]
24772     [Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]
24773      [Power Steering, A/C, Airbag, Leather Seat, Back Camera]
24774    [Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]
Name: accessories, Length: 24775, dtype: object

In [ ]:
# df["accessories"] = df["accessories"].fillna("")
# # df["accessories"] = df["accessories"].str.lower()

In [ ]:
df['accessories']

In [ ]:
# df["accessories_list"] = df["accessories"].apply(lambda x: [i.strip() for i in x.split(",") if i])

In [ ]:
# df['accessories_list']

### Clean destination_port column

In [78]:
df['destination_port']

0        Mombasa
1        Mombasa
2        Mombasa
3        Mombasa
4        Mombasa
          ...   
24770    Mombasa
24771    Mombasa
24772    Mombasa
24773    Mombasa
24774    Mombasa
Name: destination_port, Length: 24775, dtype: str

In [79]:
df["destination_port"] = df["destination_port"].fillna("Unknown")

In [87]:
df[df["destination_port"].str.contains('Unknown')]

,vehicle_id,ref_no,title,price_usd,total_price_usd,mileage,year,engine_cc,transmission,location,model_code,steering,fuel,seats,engine_code,color,drive,doors,accessories,destination_port,vehicle_url,make,model,mileage_km,shipping_cost,car_age,transmission_type
13,14865016,CC842499,2024 MG MG OTHERS,19300.0,NaN,"26,957 km",2024,NaN,AT,Thailand,NaN,Right,Electric,5.0,NaN,White,2WD,5,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",Unknown,/mg/mg-others/cc842499/id/14865016/?mfg_year_from=2018,OTHER,MG OTHERS,26957.0,NaN,2,Automatic
32,14990108,CD038472,2026 JAECOO 5 MAX,19030.0,NaN,-,2026,1500.0,AT,Thailand,NaN,Right,Electric,7.0,NaN,Other,2WD,5,"[A/C, Airbag, Keyless Entry, Back Camera, Sun Roof, Push Start, ABS]",Unknown,/jaecoo/5-max/cd038472/id/14990108/?mfg_year_from=2018,OTHER,5 MAX,NaN,NaN,0,Automatic
45,14989829,CD038232,2026 JAECOO 6 PRO,26250.0,NaN,-,2026,1600.0,AT,Thailand,NaN,Right,Electric,5.0,NaN,Gray,2WD,5,"[A/C, Airbag, Keyless Entry, Back Camera, Sun Roof, Push Start, ABS]",Unknown,/jaecoo/6-pro/cd038232/id/14989829/?mfg_year_from=2018,OTHER,6 PRO,NaN,NaN,0,Automatic
46,14989977,CD038362,2026 JAECOO 6,32820.0,NaN,-,2026,1600.0,AT,Thailand,NaN,Right,Electric,5.0,NaN,Gray,4WD,5,"[A/C, Airbag, Keyless Entry, Back Camera, Sun Roof, Push Start, ABS]",Unknown,/jaecoo/6/cd038362/id/14989977/?mfg_year_from=2018,OTHER,6,NaN,NaN,0,Automatic
48,14990206,CD038570,2026 JAECOO 5 MAX,21330.0,NaN,-,2026,1500.0,AT,Thailand,NaN,Right,Electric,5.0,NaN,Gray,2WD,5,"[A/C, Airbag, Keyless Entry, Back Camera, Sun Roof, Push Start]",Unknown,/jaecoo/5-max/cd038570/id/14990206/?mfg_year_from=2018,OTHER,5 MAX,NaN,NaN,0,Automatic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24662,15173441,CD220283,2024 NISSAN SAKURA,10990.0,NaN,"2,403 km",2024,NaN,CVT,Osaka,ZAA-B6AW,Right,Electric,NaN,NaN,White,-,5,"[Power Steering, A/C, Airbag, Keyless Entry, Alloy Wheels, Power Window, ABS]",Unknown,/nissan/sakura/cd220283/id/15173441/?mfg_year_from=2018,NISSAN,SAKURA,2403.0,NaN,2,Automatic
24699,15174000,CD221143,2025 MERCEDES-BENZ EQA,24770.0,NaN,"3,436 km",2025,NaN,AT,Yokohama,243702C,Right,Electric,5.0,NaN,White,-,5,"[Power Steering, A/C, Airbag, Keyless Entry, Alloy Wheels, Power Window, ABS]",Unknown,/mercedes-benz/eqa/cd221143/id/15174000/?mfg_year_from=2018,MERCEDES-BENZ,EQA,3436.0,NaN,1,Automatic
24737,15172887,CD219735,2022 NISSAN SAKURA G,8630.0,NaN,"19,000 km",2022,NaN,CVT,Yokohama,ZAA-B6AW,Right,Electric,4.0,MM48,Black,2WD,5,[Back Camera],Unknown,/nissan/sakura/cd219735/id/15172887/?mfg_year_from=2018,NISSAN,SAKURA G,19000.0,NaN,4,Automatic
24738,15172888,CD219736,2023 NISSAN ARIYA B9E-4ORCE,26740.0,NaN,"39,000 km",2023,NaN,CVT,Kyushu,ZAA-SNFE0,Right,Electric,5.0,AM67-AM67,Bronze,4WD,5,"[Back Camera, Sun Roof, Power Seat]",Unknown,/nissan/ariya/cd219736/id/15172888/?mfg_year_from=2018,NISSAN,ARIYA B9E-4ORCE,39000.0,NaN,3,Automatic


### Clean vehicle_url *****

In [ ]:
df['vehicle_url']

In [ ]:
df.info()

## Final Clean dataset

In [81]:
columns = [
    "make",
    "model",
    "model_code",

    "year",
    "car_age",

    "engine_cc",
    "engine_code",
    "fuel",
    "transmission_type",
    "drive",

    "mileage_km",
    "steering",
    "color",
    "seats",
    "doors",

    "price_usd",
    "total_price_usd",
    "shipping_cost",

    "location",
    "destination_port",

    "accessories",
    # "accessories",

    "vehicle_id",
    "ref_no",
    "vehicle_url"
]

In [ ]:
# clean_df = df[[
#     "make",
#     "model",
#     "model_code",
#     "year",
#     "car_age",

#     "vehicle_id",
#     "ref_no",
#     "vehicle_url",

#     "engine_cc",
#     "engine_code",
#     "fuel",
#     "transmission_type",
#     "drive",

#     "mileage_km",
#     "steering",
#     "color",
#     "seats",
#     "doors",

#     "price_usd",
#     "total_price_usd",
#     "shipping_cost",

#     "location",
#     "destination_port",

#     "accessories",
#     "accessories_list"
# ]]

In [82]:
clean_df = df[columns]
clean_df

,make,model,model_code,year,car_age,engine_cc,engine_code,fuel,transmission_type,drive,mileage_km,steering,color,seats,doors,price_usd,total_price_usd,shipping_cost,location,destination_port,accessories,vehicle_id,ref_no,vehicle_url
0,JEEP,WRANGLER UNLIMITED SAHARA,ABA-JL36L,2019,7,3600.0,G,Petrol,Automatic,4WD,94263.0,Right,Gray,5.0,5,16980.0,19896.0,2916.0,Yokohama,Mombasa,[],15018558,CD066265,/jeep/wrangler/cd066265/id/15018558/?mfg_year_from=2018
1,HONDA,ACCORD,NaN,2019,7,1500.0,NaN,Petrol,Automatic,2WD,55361.0,Right,Black,5.0,4,27740.0,29731.0,1991.0,Thailand,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",14864798,CC842281,/honda/accord/cc842281/id/14864798/?mfg_year_from=2018
2,HONDA,CITY,NaN,2024,2,1000.0,NaN,Petrol,Automatic,2WD,26954.0,Right,White,5.0,4,17190.0,18416.0,1226.0,Thailand,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",14864891,CC842374,/honda/city/cc842374/id/14864891/?mfg_year_from=2018
3,HONDA,BR-V,NaN,2024,2,1500.0,NaN,Petrol,Automatic,2WD,58522.0,Right,Black,7.0,5,22080.0,24107.0,2027.0,Thailand,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",14864903,CC842386,/honda/br-v/cc842386/id/14864903/?mfg_year_from=2018
4,HONDA,CR-V,NaN,2023,3,1500.0,NaN,Petrol,Automatic,4WD,30859.0,Right,White,7.0,5,40390.0,42560.0,2170.0,Thailand,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",14864910,CC842393,/honda/cr-v/cc842393/id/14864910/?mfg_year_from=2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24770,ISUZU,D-MAX 3.0 SX DOUBLE CAB,SX,2026,0,3000.0,NaN,Diesel,Automatic,4WD,NaN,Right,White,5.0,4,41260.0,46045.0,4785.0,Australia,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Keyless Entry]",15172794,CD219642,/isuzu/d-max/cd219642/id/15172794/?mfg_year_from=2018
24771,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2026,0,3000.0,NaN,Diesel,Automatic,4WD,NaN,Right,Silver,5.0,4,50090.0,54775.0,4685.0,Australia,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]",15172792,CD219641,/isuzu/d-max/cd219641/id/15172792/?mfg_year_from=2018
24772,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2026,0,3000.0,NaN,Diesel,Automatic,4WD,NaN,Right,Silver,4.0,4,54520.0,59205.0,4685.0,Australia,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Alloy Wheels]",15172791,CD219640,/isuzu/d-max/cd219640/id/15172791/?mfg_year_from=2018
24773,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2025,1,3000.0,NaN,Diesel,Automatic,4WD,NaN,Right,Black,5.0,4,51910.0,56595.0,4685.0,Australia,Mombasa,"[Power Steering, A/C, Airbag, Leather Seat, Back Camera]",15172790,CD219639,/isuzu/d-max/cd219639/id/15172790/?mfg_year_from=2018


In [83]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24775 entries, 0 to 24774
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   make               24775 non-null  str    
 1   model              24775 non-null  str    
 2   model_code         22116 non-null  str    
 3   year               24775 non-null  int64  
 4   car_age            24775 non-null  int64  
 5   engine_cc          24552 non-null  float64
 6   engine_code        1303 non-null   str    
 7   fuel               24775 non-null  str    
 8   transmission_type  24775 non-null  str    
 9   drive              24775 non-null  str    
 10  mileage_km         23637 non-null  float64
 11  steering           24775 non-null  str    
 12  color              24775 non-null  str    
 13  seats              23901 non-null  float64
 14  doors              24566 non-null  str    
 15  price_usd          24744 non-null  float64
 16  total_price_usd    24568 non-null

## Save clean dataset

In [84]:
clean_df.to_csv("../data/clean/5_beforward_clean_data.csv", index=False)